# Import Correlation CSV Files

In [ ]:
# Import all correlation matrix CSV files from the folder
import os
import pandas as pd

correlation_folder = 'correlation_matrices_all_subjects'

# List all CSV files in the folder
correlation_files = [
    os.path.join(correlation_folder, f)
    for f in os.listdir(correlation_folder)
    if f.endswith('.csv')
]

# Read all correlation matrices into a list of DataFrames
correlation_matrices = [pd.read_csv(f, index_col=0) for f in correlation_files]

print(f"Imported {len(correlation_matrices)} correlation matrices.")

Imported 129 correlation matrices.


# Convert To Binary Adjacency Matrix

In [2]:
# Convert correlation matrices to binary adjacency matrices using threshold = 0.3
threshold = 0.3

binary_adjacency_matrices = [
    (matrix > threshold).astype(int)
    for matrix in correlation_matrices
]

print(f"Converted {len(binary_adjacency_matrices)} matrices to binary adjacency matrices with threshold {threshold}.")

Converted 129 matrices to binary adjacency matrices with threshold 0.3.


# Calculate Jaccard Index For All Pairs

In [3]:
# Calculate Jaccard index for all pairs of binary adjacency matrices
import numpy as np

n = len(binary_adjacency_matrices)
jaccard_matrix = np.zeros((n, n))

for i in range(n):
    a_flat = binary_adjacency_matrices[i].values.flatten()
    for j in range(n):
        b_flat = binary_adjacency_matrices[j].values.flatten()
        intersection = np.logical_and(a_flat, b_flat).sum()
        union = np.logical_or(a_flat, b_flat).sum()
        jaccard_matrix[i, j] = intersection / union if union != 0 else 0

print("Jaccard index matrix shape:", jaccard_matrix.shape)
jaccard_matrix

Jaccard index matrix shape: (129, 129)


array([[1.        , 0.46863469, 0.82698962, ..., 0.79061372, 0.72563177,
        0.72363636],
       [0.46863469, 1.        , 0.50185874, ..., 0.4939759 , 0.54585153,
        0.51515152],
       [0.82698962, 0.50185874, 1.        , ..., 0.78647687, 0.76      ,
        0.74545455],
       ...,
       [0.79061372, 0.4939759 , 0.78647687, ..., 1.        , 0.77254902,
        0.65313653],
       [0.72563177, 0.54585153, 0.76      , ..., 0.77254902, 1.        ,
        0.66023166],
       [0.72363636, 0.51515152, 0.74545455, ..., 0.65313653, 0.66023166,
        1.        ]], shape=(129, 129))

# Create A DataFrame And Save The Results

In [ ]:
# Extract subject-session labels from the filenames (remove folder and extension)
labels = [os.path.splitext(os.path.basename(f))[0] for f in correlation_files]

# Create a DataFrame with labels for both rows and columns
jaccard_df = pd.DataFrame(jaccard_matrix, index=labels, columns=labels)

# Save the DataFrame as a CSV file
output_csv = 'jaccard_index_matrix_labeled.csv'
jaccard_df.to_csv(output_csv)
print(f"Jaccard index matrix with labels saved as '{output_csv}'")

Jaccard index matrix with labels saved as 'jaccard_index_matrix_labeled.csv'


# Remove The Diagonal and The Upper Triangular Part

In [11]:
# Convert the Jaccard index DataFrame to its lower triangular matrix
lower_triangular_jaccard_df = jaccard_df.where(np.tril(np.ones(jaccard_df.shape), k=0).astype(bool))

# Remove diagonal elements from the lower triangular Jaccard index DataFrame
lower_triangular_no_diag_jaccard_df = lower_triangular_jaccard_df.copy()
np.fill_diagonal(lower_triangular_no_diag_jaccard_df.values, np.nan)

# Display the Annotated matrix
lower_triangular_no_diag_jaccard_df.head()

,sub-NORB00064_ses-2_correlation_matrix_avg,sub-NORB00084_ses-1_correlation_matrix_avg,sub-NORB00015_ses-1_correlation_matrix_avg,sub-NORB00049_ses-1_correlation_matrix_avg,sub-NORB00029_ses-1_correlation_matrix_avg,sub-NORB00086_ses-2_correlation_matrix_avg,sub-NORB00098_ses-1_correlation_matrix_avg,sub-NORB00034_ses-1_correlation_matrix_avg,sub-NORB00027_ses-1_correlation_matrix_avg,sub-NORB00065_ses-2_correlation_matrix_avg,...,sub-NORB00039_ses-1_correlation_matrix_avg,sub-NORB00048_ses-1_correlation_matrix_avg,sub-NORB00046_ses-1_correlation_matrix_avg,sub-NORB00066_ses-1_correlation_matrix_avg,sub-NORB00100_ses-1_correlation_matrix_avg,sub-NORB00051_ses-1_correlation_matrix_avg,sub-NORB00057_ses-1_correlation_matrix_avg,sub-NORB00078_ses-2_correlation_matrix_avg,sub-NORB00067_ses-1_correlation_matrix_avg,sub-NORB00017_ses-1_correlation_matrix_avg
sub-NORB00064_ses-2_correlation_matrix_avg,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sub-NORB00084_ses-1_correlation_matrix_avg,0.468635,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sub-NORB00015_ses-1_correlation_matrix_avg,0.826990,0.501859,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sub-NORB00049_ses-1_correlation_matrix_avg,0.814696,0.436893,0.833866,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
sub-NORB00029_ses-1_correlation_matrix_avg,0.761404,0.494071,0.874539,0.762058,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN


# Top 30 Pairs

In [ ]:
# Stack the lower triangular matrix (excluding diagonal) into a long format
pairs = []
for i in range(1, lower_triangular_no_diag_jaccard_df.shape[0]):
    for j in range(i):
        value = lower_triangular_no_diag_jaccard_df.iloc[i, j]
        if not np.isnan(value):
            pairs.append((lower_triangular_no_diag_jaccard_df.index[i], lower_triangular_no_diag_jaccard_df.columns[j], value))

# Sort pairs by Jaccard index value (descending)
top_pairs = sorted(pairs, key=lambda x: x[2], reverse=True)[:30]

# Print the top 30 pairs
print("Top 30 subject-session pairs by Jaccard index:")
for subj1, subj2, val in top_pairs:
    print(f"{subj1} vs {subj2}: {val:.4f}")

Top 30 subject-session pairs by Jaccard index:
sub-NORB00079_ses-2_correlation_matrix_avg vs sub-NORB00013_ses-1_correlation_matrix_avg: 1.0000
sub-NORB00024_ses-1_correlation_matrix_avg vs sub-NORB00013_ses-1_correlation_matrix_avg: 1.0000
sub-NORB00024_ses-1_correlation_matrix_avg vs sub-NORB00079_ses-2_correlation_matrix_avg: 1.0000
sub-NORB00025_ses-1_correlation_matrix_avg vs sub-NORB00013_ses-1_correlation_matrix_avg: 0.9834
sub-NORB00079_ses-2_correlation_matrix_avg vs sub-NORB00025_ses-1_correlation_matrix_avg: 0.9834
sub-NORB00024_ses-1_correlation_matrix_avg vs sub-NORB00025_ses-1_correlation_matrix_avg: 0.9834
sub-NORB00081_ses-1_correlation_matrix_avg vs sub-NORB00035_ses-1_correlation_matrix_avg: 0.9535
sub-NORB00020_ses-1_correlation_matrix_avg vs sub-NORB00013_ses-1_correlation_matrix_avg: 0.9501
sub-NORB00020_ses-1_correlation_matrix_avg vs sub-NORB00079_ses-2_correlation_matrix_avg: 0.9501
sub-NORB00024_ses-1_correlation_matrix_avg vs sub-NORB00020_ses-1_correlation_ma

# Visualisation